In [1]:
import os
import json
import csv
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, Optional, Literal, Any

import numpy as np
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

import datasets
import unet
import unetFixed
from SplitNet import SplitNet
import metrics

import json

with open("dataset_priors.json", "r") as f:
    priors = json.load(f)


# -----------------------------------------------------------------------------
# Global setup
# -----------------------------------------------------------------------------

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------------------------------------------------------
# Type aliases
# -----------------------------------------------------------------------------

ModelType = Literal[
    "splitnet_attn",
    "splitnet",
    "unet",
    "attn_unet",
    "unetfixed",
    "unetFixed",
]

DatasetMode = Literal[
    "border",
    "fixed",
]

TrainingMode = Literal[
    "physics_limited",
    "baseline_full",
]


# -----------------------------------------------------------------------------
# Experiment configuration
# -----------------------------------------------------------------------------

@dataclass
class ExperimentConfig:
    model_type: ModelType = "splitnet_attn"
    dataset_mode: DatasetMode = "fixed"
    training_mode: TrainingMode = "physics_limited"

    loss_config: Optional[Dict[str, Dict[str, Any]]] = None

    epochs: int = 50
    batch_size: int = 8
    lr: float = 1e-3

    channels: str = "KP"

    train_sims_path: str = "../train_sims.npy"
    val_sims_path: str = "../val_sims.npy"
    sim_max_exclusive: Optional[int] = 500

    save_dir: str = "minimum_info/results"
    run_name: Optional[str] = None
    save_prefix: Optional[str] = None
    save_best: bool = True

    # Use keys from evaluate_loader:
    # "supervised_mse", "total_mse", "ssim", "psnr", etc.
    best_metric: Optional[str] = None

    dataset_kwargs: Optional[Dict[str, Any]] = None


# -----------------------------------------------------------------------------
# Loss configuration helpers
# -----------------------------------------------------------------------------

def default_loss_config(training_mode: TrainingMode):
    """
    Default behavior matching the old loader.

    baseline_full:
        MSE over whole image.

    physics_limited:
        MSE over dataset mask + Darcy over whole prediction.
    """
    if training_mode == "baseline_full":
        return {
            "mse": {
                "weight": 1.0,
                "region": "all",
            }
        }

    if training_mode == "physics_limited":
        return {
            "mse": {
                "weight": 1.0,
                "region": "mask",
            },
            "darcy": {
                "weight": 1.0,
                "region": "all",
            },
        }

    raise ValueError(f"Unknown training_mode: {training_mode}")


def old_darcy_loss_config(training_mode: TrainingMode, darcy_weight: float):
    """
    Backwards-compatible helper for old Darcy sweeps.
    """
    if training_mode == "baseline_full":
        return {
            "mse": {
                "weight": 1.0,
                "region": "all",
            }
        }

    if training_mode == "physics_limited":
        return {
            "mse": {
                "weight": 1.0,
                "region": "mask",
            },
            "darcy": {
                "weight": darcy_weight,
                "region": "all",
            },
        }

    raise ValueError(f"Unknown training_mode: {training_mode}")


def compute_weighted_loss(out, label, mask, loss_config):
    total_loss = torch.tensor(0.0, device=out.device)

    label = metrics.align_channels(label, out)

    for metric_name, cfg in loss_config.items():
        cfg = cfg.copy()
        weight = cfg.pop("weight", 1.0)

        metric_value = metrics.compute_metric(
            metric_name,
            pred=out,
            target=label,
            mask=mask,
            **cfg,
        )

        total_loss = total_loss + weight * metric_value

    return total_loss


def make_loss_name(loss_config):
    parts = []

    for metric_name, cfg in loss_config.items():
        weight = cfg.get("weight", 1.0)
        region = cfg.get("region", "all")
        channel = cfg.get("channel", None)

        safe_weight = str(weight).replace(".", "p")

        part = f"{metric_name}_{safe_weight}_{region}"

        if channel is not None:
            part += f"_ch{channel}"

        parts.append(part)

    return "__".join(parts)


def metric_should_maximize(metric_name):
    return metric_name in ["ssim", "psnr"]


# -----------------------------------------------------------------------------
# Model factory
# -----------------------------------------------------------------------------

def get_num_channels(channels: str) -> int:
    if channels == "all":
        return 3
    if channels == "KP":
        return 2
    if channels in ["K", "P", "phi"]:
        return 1

    raise ValueError("channels must be 'all', 'KP', 'K', 'P', or 'phi'.")


def normalize_model_type(model_type: str) -> str:
    model_type = model_type.lower()

    if model_type in ["unetfixed", "unetfixed", "prof_unet"]:
        return "unetfixed"

    return model_type


def make_model(model_type: ModelType = "splitnet_attn", channels: str = "KP") -> nn.Module:
    model_type = normalize_model_type(model_type)
    num_channels = get_num_channels(channels)

    if model_type == "splitnet_attn":
        if channels != "KP":
            raise ValueError("SplitNet only supports channels='KP'.")
        return SplitNet(attn=True).to(DEVICE)

    if model_type == "splitnet":
        if channels != "KP":
            raise ValueError("SplitNet only supports channels='KP'.")
        return SplitNet(attn=False).to(DEVICE)

    if model_type == "unet":
        return unet.SmallUnet(channels=num_channels).to(DEVICE)

    if model_type == "attn_unet":
        return unet.AttnUnet(channels=num_channels).to(DEVICE)

    if model_type == "unetfixed":
        return unetFixed.UNet(
            in_channels=num_channels,
            num_classes=num_channels,
        ).to(DEVICE)

    raise ValueError(f"Unknown model_type: {model_type}")


# -----------------------------------------------------------------------------
# Dataset selection
# -----------------------------------------------------------------------------

def get_dataset_class(dataset_mode: DatasetMode, training_mode: TrainingMode):
    if training_mode == "physics_limited":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetLimited
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetLimited

    if training_mode == "baseline_full":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetFull
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetFull

    raise ValueError(
        f"Invalid dataset/training combination: "
        f"dataset_mode={dataset_mode}, training_mode={training_mode}"
    )


def load_sim_ids(path: str, sim_max_exclusive: Optional[int]) -> np.ndarray:
    sims = np.load(path)

    if sim_max_exclusive is not None:
        sims = sims[sims < sim_max_exclusive]

    return sims


class ChannelSelectDataset(torch.utils.data.Dataset):
    """
    Safety wrapper that ensures the dataset returns requested channels.
    """

    def __init__(self, base_dataset, channels: str = "KP"):
        self.base_dataset = base_dataset
        self.channels = channels

    def _idx(self):
        if self.channels == "all":
            return [0, 1, 2]
        if self.channels == "KP":
            return [0, 1]
        if self.channels == "K":
            return [0]
        if self.channels == "P":
            return [1]
        if self.channels == "phi":
            return [2]

        raise ValueError(f"Bad channels: {self.channels}")

    def __len__(self):
        return len(self.base_dataset)

    def __getattr__(self, name):
        return getattr(self.base_dataset, name)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        chans = self._idx()

        if len(item) == 3:
            feat, label, mask = item
            return feat[chans], label[chans], mask

        feat, label = item
        return feat[chans], label[chans]


def make_loaders(config: ExperimentConfig) -> Tuple[DataLoader, DataLoader]:
    train_sims = load_sim_ids(config.train_sims_path, config.sim_max_exclusive)
    val_sims = load_sim_ids(config.val_sims_path, config.sim_max_exclusive)

    dataset_cls = get_dataset_class(config.dataset_mode, config.training_mode)

    kwargs = dict(config.dataset_kwargs or {})
    kwargs["channels"] = config.channels

    train_data = dataset_cls(train_sims, **kwargs)
    val_data = dataset_cls(val_sims, **kwargs)

    train_data = ChannelSelectDataset(train_data, config.channels)
    val_data = ChannelSelectDataset(val_data, config.channels)

    train_loader = DataLoader(
        train_data,
        batch_size=config.batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_data,
        batch_size=config.batch_size,
        shuffle=False,
    )

    return train_loader, val_loader


def unpack_batch(batch, training_mode: TrainingMode):
    if training_mode == "physics_limited":
        feat, label, mask = batch
        return feat, label, mask

    if training_mode == "baseline_full":
        feat, label = batch
        return feat, label, None

    raise ValueError(f"Unknown training_mode: {training_mode}")


# -----------------------------------------------------------------------------
# Saving helpers
# -----------------------------------------------------------------------------

def make_run_prefix(config: ExperimentConfig) -> str:
    if config.save_prefix:
        return config.save_prefix

    if config.run_name:
        name = config.run_name
    else:
        loss_name = make_loss_name(
            config.loss_config or default_loss_config(config.training_mode)
        )
        name = f"{config.dataset_mode}_{config.training_mode}_{config.model_type}_{loss_name}"

    return os.path.join(config.save_dir, name)


def ensure_parent_dir(path_prefix: str):
    folder = os.path.dirname(path_prefix)
    if folder:
        os.makedirs(folder, exist_ok=True)


def save_json(path: str, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def save_history_csv(path: str, history: Dict[str, Any]):
    curve_keys = [
        "train_loss_used",

        "train_total_mse",
        "train_supervised_mse",
        "train_mask_mse",
        "train_nonmask_mse",
        "train_darcy",
        "train_darcy_match",
        "train_ssim",
        "train_psnr",
        "train_high_grad_mse_k",
        "train_high_grad_mse_p",

        "val_total_mse",
        "val_supervised_mse",
        "val_mask_mse",
        "val_nonmask_mse",
        "val_darcy",
        "val_darcy_match",
        "val_ssim",
        "val_psnr",
        "val_high_grad_mse_k",
        "val_high_grad_mse_p",
    ]

    n_epochs = len(history["train_loss_used"])

    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch"] + curve_keys)

        for i in range(n_epochs):
            row = [i + 1]
            for key in curve_keys:
                values = history.get(key, [])
                row.append(values[i] if i < len(values) else "")
            writer.writerow(row)


def make_run_summary(
    history: Dict[str, Any],
    config: ExperimentConfig,
    best_epoch: int,
    best_val_score: float,
) -> Dict[str, Any]:

    def clean_values(key):
        return [
            v for v in history.get(key, [])
            if v == v and v != float("inf") and v != float("-inf")
        ]

    def best(key):
        values = clean_values(key)
        return min(values) if values else None

    def highest(key):
        values = clean_values(key)
        return max(values) if values else None

    def final(key):
        values = clean_values(key)
        return values[-1] if values else None

    return {
        "run_name": config.run_name,
        "model_type": config.model_type,
        "dataset_mode": config.dataset_mode,
        "training_mode": config.training_mode,
        "channels": config.channels,
        "loss_config": config.loss_config,
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "lr": config.lr,

        "best_epoch": best_epoch,
        "best_val_score_used": best_val_score,

        "best_train_loss_used": best("train_loss_used"),

        "best_val_total_mse": best("val_total_mse"),
        "best_val_supervised_mse": best("val_supervised_mse"),
        "best_val_mask_mse": best("val_mask_mse"),
        "best_val_nonmask_mse": best("val_nonmask_mse"),
        "best_val_darcy": best("val_darcy"),
        "best_val_darcy_match": best("val_darcy_match"),
        "best_val_ssim": highest("val_ssim"),
        "best_val_psnr": highest("val_psnr"),
        "best_val_high_grad_mse_k": best("val_high_grad_mse_k"),
        "best_val_high_grad_mse_p": best("val_high_grad_mse_p"),

        "final_train_loss_used": final("train_loss_used"),
        "final_val_total_mse": final("val_total_mse"),
        "final_val_supervised_mse": final("val_supervised_mse"),
        "final_val_mask_mse": final("val_mask_mse"),
        "final_val_nonmask_mse": final("val_nonmask_mse"),
        "final_val_darcy": final("val_darcy"),
        "final_val_darcy_match": final("val_darcy_match"),
        "final_val_ssim": final("val_ssim"),
        "final_val_psnr": final("val_psnr"),
        "final_val_high_grad_mse_k": final("val_high_grad_mse_k"),
        "final_val_high_grad_mse_p": final("val_high_grad_mse_p"),

        "config": asdict(config),
    }


def save_run_outputs(
    path_prefix: str,
    model: nn.Module,
    history: Dict[str, Any],
    config: ExperimentConfig,
    best_epoch: int,
    best_val_score: float,
):
    ensure_parent_dir(path_prefix)

    save_history_csv(f"{path_prefix}_history.csv", history)
    torch.save(history, f"{path_prefix}_history.pt")

    summary = make_run_summary(history, config, best_epoch, best_val_score)
    save_json(f"{path_prefix}_summary.json", summary)

    torch.save(model.state_dict(), f"{path_prefix}_final_state.pt")


# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

def safe_compute_metric(name, pred, target=None, mask=None, default=float("nan"), **kwargs):
    """
    Some metrics require KP channels. This avoids crashing evaluation
    when running single-channel models.
    """
    try:
        return metrics.compute_metric(
            name,
            pred=pred,
            target=target,
            mask=mask,
            **kwargs,
        ).item()
    except ValueError:
        return default


def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    config: ExperimentConfig,
) -> Dict[str, float]:

    model.eval()

    metric_sums = {
        "total_mse": 0.0,
        "supervised_mse": 0.0,
        "mask_mse": 0.0,
        "nonmask_mse": 0.0,
        "darcy": 0.0,
        "darcy_match": 0.0,
        "ssim": 0.0,
        "psnr": 0.0,
        "high_grad_mse_k": 0.0,
        "high_grad_mse_p": 0.0,
    }

    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)

            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            out = model(feat)
            label = metrics.align_channels(label, out)

            metric_sums["total_mse"] += metrics.compute_metric(
                "mse", out, label, region="all"
            ).item()

            if config.training_mode == "physics_limited":
                metric_sums["supervised_mse"] += metrics.compute_metric(
                    "mse", out, label, mask=mask, region="mask"
                ).item()
            else:
                metric_sums["supervised_mse"] += metrics.compute_metric(
                    "mse", out, label, region="all"
                ).item()

            if mask is not None:
                metric_sums["mask_mse"] += metrics.compute_metric(
                    "mse", out, label, mask=mask, region="mask"
                ).item()

                metric_sums["nonmask_mse"] += metrics.compute_metric(
                    "mse", out, label, mask=mask, region="nonmask"
                ).item()
            else:
                metric_sums["mask_mse"] += float("nan")
                metric_sums["nonmask_mse"] += float("nan")

            metric_sums["darcy"] += safe_compute_metric(
                "darcy", out, region="all"
            )

            metric_sums["darcy_match"] += safe_compute_metric(
                "darcy_match", out, label, region="all"
            )

            metric_sums["ssim"] += metrics.compute_metric(
                "ssim", out, label, region="all"
            ).item()

            metric_sums["psnr"] += metrics.compute_metric(
                "psnr", out, label, region="all"
            ).item()

            metric_sums["high_grad_mse_k"] += metrics.compute_metric(
                "mse", out, label, region="high_gradient", channel=0
            ).item()

            if out.shape[1] > 1:
                metric_sums["high_grad_mse_p"] += metrics.compute_metric(
                    "mse", out, label, region="high_gradient", channel=1
                ).item()
            else:
                metric_sums["high_grad_mse_p"] += float("nan")

            n_batches += 1

    return {
        key: value / n_batches
        for key, value in metric_sums.items()
    }


# -----------------------------------------------------------------------------
# Training
# -----------------------------------------------------------------------------

def run_experiment(**kwargs):
    config = ExperimentConfig(**kwargs)

    if config.loss_config is None:
        config.loss_config = default_loss_config(config.training_mode)

    path_prefix = make_run_prefix(config)
    ensure_parent_dir(path_prefix)

    model = make_model(config.model_type, config.channels)
    optimizer = Adam(model.parameters(), lr=config.lr)

    train_loader, val_loader = make_loaders(config)

    history = {
        "train_loss_used": [],

        "train_total_mse": [],
        "train_supervised_mse": [],
        "train_mask_mse": [],
        "train_nonmask_mse": [],
        "train_darcy": [],
        "train_darcy_match": [],
        "train_ssim": [],
        "train_psnr": [],
        "train_high_grad_mse_k": [],
        "train_high_grad_mse_p": [],

        "val_total_mse": [],
        "val_supervised_mse": [],
        "val_mask_mse": [],
        "val_nonmask_mse": [],
        "val_darcy": [],
        "val_darcy_match": [],
        "val_ssim": [],
        "val_psnr": [],
        "val_high_grad_mse_k": [],
        "val_high_grad_mse_p": [],

        "config": asdict(config),
    }

    metric_for_best = config.best_metric or (
        "supervised_mse" if config.training_mode == "physics_limited" else "total_mse"
    )

    maximize_best = metric_should_maximize(metric_for_best)
    best_val_score = float("-inf") if maximize_best else float("inf")
    best_epoch = 0

    for epoch in tqdm(range(1, config.epochs + 1)):
        model.train()

        epoch_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)

            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            optimizer.zero_grad()

            out = model(feat)
            label = metrics.align_channels(label, out)

            loss = compute_weighted_loss(
                out=out,
                label=label,
                mask=mask,
                loss_config=config.loss_config,
            )

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        history["train_loss_used"].append(epoch_loss / n_batches)

        train_metrics = evaluate_loader(model, train_loader, config)
        val_metrics = evaluate_loader(model, val_loader, config)

        for split, metric_dict in [("train", train_metrics), ("val", val_metrics)]:
            for key, value in metric_dict.items():
                history[f"{split}_{key}"].append(value)

        val_score = val_metrics[metric_for_best]

        is_better = (
            val_score > best_val_score
            if maximize_best
            else val_score < best_val_score
        )

        if is_better:
            best_val_score = val_score
            best_epoch = epoch

            if config.save_best:
                torch.save(model.state_dict(), f"{path_prefix}_best_state.pt")

    print(f"Best epoch: {best_epoch}, best val_{metric_for_best}: {best_val_score:.6f}")

    save_run_outputs(path_prefix, model, history, config, best_epoch, best_val_score)

    return model, history


def run_experiment_grid(
    model_types=("splitnet_attn",),
    dataset_modes=("fixed",),
    training_modes=("physics_limited",),
    loss_configs=None,
    darcy_weights=None,
    epochs=50,
    base_save_dir="results",
    **kwargs,
):
    """
    Runs every combination of:

        model_type x dataset_mode x training_mode x loss_config

    New style:
        Pass loss_configs as a dict or list.

    Backwards-compatible old style:
        Pass darcy_weights=(0.1, 1.0, 10.0)
        and do not pass loss_configs.
    """

    results = {}

    for dataset_mode in dataset_modes:
        for model_type in model_types:
            for training_mode in training_modes:

                if loss_configs is not None:
                    if isinstance(loss_configs, dict):
                        configs_to_run = loss_configs
                    elif isinstance(loss_configs, list):
                        configs_to_run = {
                            f"loss_{i + 1}": lc
                            for i, lc in enumerate(loss_configs)
                        }
                    else:
                        raise ValueError("loss_configs must be None, dict, or list.")

                else:
                    if darcy_weights is not None:
                        weights = [0.0] if training_mode == "baseline_full" else darcy_weights

                        configs_to_run = {}

                        for w in weights:
                            safe_w = str(w).replace(".", "p")

                            if training_mode == "baseline_full":
                                name = "mse_all"
                            else:
                                name = f"mse_mask_darcy_{safe_w}"

                            configs_to_run[name] = old_darcy_loss_config(training_mode, w)

                    else:
                        configs_to_run = {
                            "default": default_loss_config(training_mode)
                        }

                for loss_name, loss_config in configs_to_run.items():

                    if training_mode == "baseline_full":
                        for metric_name, cfg in loss_config.items():
                            if cfg.get("region", "all") == "mask":
                                raise ValueError(
                                    f"baseline_full has no mask, but loss "
                                    f"'{metric_name}' uses region='mask'."
                                )

                    safe_loss_name = (
                        make_loss_name(loss_config)
                        if loss_name == "default"
                        else loss_name
                    )

                    run_name = (
                        f"{dataset_mode}_{model_type}_{training_mode}_{safe_loss_name}"
                    )

                    print(f"\n===== Running {run_name} =====")
                    print(f"Loss config: {loss_config}")

                    model, history = run_experiment(
                        model_type=model_type,
                        dataset_mode=dataset_mode,
                        training_mode=training_mode,
                        loss_config=loss_config,
                        epochs=epochs,
                        save_dir=base_save_dir,
                        run_name=run_name,
                        **kwargs,
                    )

                    results[run_name] = {
                        "model": model,
                        "history": history,
                        "loss_config": loss_config,
                    }

    return results

In [2]:
loss_configs_stats = {
    "mask_mse_darcy_tvP_stats": {
        "mse": {
            "weight": 1.0,
            "region": "mask",
        },
        "darcy": {
            "weight": 5.0,
            "region": "all",
        },
        "tv": {
            "weight": 0.01,
            "region": "all",
            "channel": 1,   # pressure smoothness
        },
        "channel_stats": {
            "weight": 0.1,
            "region": "all",
            "channel_mean": priors["channel_mean"][:2],
            "channel_std": priors["channel_std"][:2],
        },
    }
}

results_stats = run_experiment_grid(
    model_types=("splitnet",),
    dataset_modes=("border",),
    training_modes=("physics_limited",),
    loss_configs=loss_configs_stats,
    epochs=20,
    sim_max_exclusive=400,
    base_save_dir="limited_prior_tests",
)


loss_configs_lowk = {
    "mask_mse_darcy_lowKarea": {
        "mse": {
            "weight": 1.0,
            "region": "mask",
        },
        "darcy": {
            "weight": 5.0,
            "region": "all",
        },
        "low_k_area": {
            "weight": 0.1,
            "region": "all",
            "target_area": priors["low_k_area_mean"],
            "threshold": priors["low_k_threshold"],
            "channel": 0,
        },
    }
}

results_lowk = run_experiment_grid(
    model_types=("splitnet",),
    dataset_modes=("border",),
    training_modes=("physics_limited",),
    loss_configs=loss_configs_lowk,
    epochs=20,
    sim_max_exclusive=400,
    base_save_dir="limited_prior_tests",
)


plain_limited = run_experiment_grid(
    model_types=("splitnet",),
    dataset_modes=("border",),
    training_modes=("physics_limited",),
    darcy_weights=(5.0,),
    epochs=20,
    sim_max_exclusive=400,
    base_save_dir="limited_prior_tests",
)

plain_limited = run_experiment_grid(
    model_types=("splitnet",),
    dataset_modes=("border",),
    training_modes=("baseline_full",),
    darcy_weights=(0.0,),
    epochs=20,
    sim_max_exclusive=400,
    base_save_dir="limited_prior_tests",
)



===== Running border_splitnet_physics_limited_mask_mse_darcy_tvP_stats =====
Loss config: {'mse': {'weight': 1.0, 'region': 'mask'}, 'darcy': {'weight': 5.0, 'region': 'all'}, 'tv': {'weight': 0.01, 'region': 'all', 'channel': 1}, 'channel_stats': {'weight': 0.1, 'region': 'all', 'channel_mean': [-0.18784285351201244, -0.6110295218298076], 'channel_std': [0.2515213950708161, 0.3208698303289577]}}


  5%|▌         | 1/20 [11:51<3:45:24, 711.84s/it]


KeyboardInterrupt: 